In [1]:
!pip -q install -U "transformers>=4.48.0" datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.0 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

# Đường dẫn trỏ thẳng tới checkpoint tốt nhất của MNLI bạn vừa chạy xong
MODEL_NAME = "/kaggle/input/notebooks/liennnnnn/mnli-modernbert-base/modernbert_base_mnli/best_mnli_model"
TASK_NAME = "rte"

# Tham số tối ưu cho RTE từ bài báo
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 3

# Cấu hình tối ưu cho Kaggle T4
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1 
MAX_LENGTH = 512
SEED = 42
OUTPUT_DIR = "./modernbert_base_rte_from_mnli"

set_seed(SEED)

In [3]:
# Tải tập dữ liệu RTE từ nguồn chuẩn nyu-mll
raw_datasets = load_dataset("nyu-mll/glue", TASK_NAME)
print(raw_datasets)
print("Ví dụ tập Train của RTE:", raw_datasets["train"][0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

rte/train-00000-of-00001.parquet:   0%|          | 0.00/584k [00:00<?, ?B/s]

rte/validation-00000-of-00001.parquet:   0%|          | 0.00/69.0k [00:00<?, ?B/s]

rte/test-00000-of-00001.parquet:   0%|          | 0.00/621k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 2490
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 277
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3000
    })
})
Ví dụ tập Train của RTE: {'sentence1': 'No Weapons of Mass Destruction Found in Iraq Yet.', 'sentence2': 'Weapons of Mass Destruction Found in Iraq.', 'label': 1, 'idx': 0}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    # Dữ liệu RTE gồm 2 trường văn bản: sentence1 và sentence2
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

keep_columns = ["input_ids", "attention_mask", "labels"]
tokenized_datasets = tokenized_datasets.remove_columns(
    [col for col in tokenized_datasets["train"].column_names if col not in keep_columns]
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

Map:   0%|          | 0/277 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [5]:
label_names = raw_datasets["train"].features["label"].names
num_labels = len(label_names)
print("Nhãn của RTE:", label_names) # Sẽ hiển thị ['entailment', 'not_entailment']

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

# Sử dụng ignore_mismatched_sizes=True để chuyển đổi từ head 3 nhãn sang head 2 nhãn
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True 
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Nhãn của RTE: ['entailment', 'not_entailment']


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/liennnnnn/mnli-modernbert-base/modernbert_base_mnli/best_mnli_model
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [6]:
glue_metric = evaluate.load("glue", TASK_NAME)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return glue_metric.compute(predictions=predictions, references=labels)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to="none",
    save_total_limit=1,
)

In [7]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Tập dev/validation của RTE
    processing_class=tokenizer, # Cập nhật theo chuẩn mới của Transformers
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Bắt đầu huấn luyện RTE chuyển giao từ MNLI...")
trainer.train()

# Đánh giá lại kết quả cuối cùng trên tập Dev
eval_result = trainer.evaluate()
rte_accuracy = eval_result["eval_accuracy"] * 100
print(f"\n==================================================")
print(f"RTE Dev Accuracy đạt được: {rte_accuracy:.2f}%")
print(f"Mục tiêu trong bài báo: ~87.40%")
print(f"==================================================")

# Lưu lại mô hình RTE hoàn chỉnh
best_rte_dir = os.path.join(OUTPUT_DIR, "best_rte_model")
trainer.save_model(best_rte_dir)
print(f"Đã lưu mô hình RTE tốt nhất tại: {best_rte_dir}")

Bắt đầu huấn luyện RTE chuyển giao từ MNLI...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.738296,0.675857,0.833935
2,0.094863,1.788396,0.855596
3,0.000881,1.773302,0.877256


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy
0.000881,1.773302,3,0.877256



RTE Dev Accuracy đạt được: 87.73%
Mục tiêu trong bài báo: ~87.40%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình RTE tốt nhất tại: ./modernbert_base_rte_from_mnli/best_rte_model
